# XGBoost

In [ ]:
import numpy as np, torch, xgboost as xgb, joblib
from sklearn.metrics import log_loss
from E_Evaluation.helpers.core import buy_metrics, predict_probs_booster

# -- Config --
NUM_BOOST_ROUND      = 10000
EARLY_STOPPING_ROUNDS = PATIENCE

def loader_to_numpy(loader):
    Xs, ys = zip(*[(xb.numpy(), yb.numpy()) for xb, yb in loader])
    return np.concatenate(Xs), np.concatenate(ys)

X_train, y_train = loader_to_numpy(train_loader)
X_val,   y_val   = loader_to_numpy(val_loader)
X_test,  y_test  = loader_to_numpy(test_loader)

num_pos, num_neg = float((y_train == 1).sum()), float((y_train == 0).sum())
scale_pos_weight = num_neg / max(1.0, num_pos)
print(f"Train {X_train.shape}  pos={int(num_pos)} neg={int(num_neg)}")
print(f"Val   {X_val.shape}    pos={int((y_val==1).sum())} neg={int((y_val==0).sum())}")
print(f"Test  {X_test.shape}   pos={int((y_test==1).sum())} neg={int((y_test==0).sum())}")
print(f"scale_pos_weight = {scale_pos_weight:.4f}")

GPU_REQUIRED_MSG        = "GPU required but torch.cuda.is_available() is False."
GPU_XGB_UNAVAILABLE_MSG = "GPU required but XGBoost CUDA training is unavailable. Install GPU-enabled XGBoost/CUDA or run on a GPU machine."
GPU_XGB_PARAMS          = {"tree_method": "hist", "device": "cuda"}

def assert_cuda_available():
    assert torch.cuda.is_available(), GPU_REQUIRED_MSG

def assert_xgb_cuda_training_available(X, y):
    dtmp = xgb.DMatrix(X[:min(2048, len(X))], label=y[:min(2048, len(y))])
    try:
        xgb.train({"objective": "binary:logistic", "eval_metric": "logloss", "max_depth": 1, "eta": 0.3, **GPU_XGB_PARAMS}, dtmp, num_boost_round=1, verbose_eval=False)
    except Exception as e:
        raise AssertionError(GPU_XGB_UNAVAILABLE_MSG) from e

assert_cuda_available()
assert_xgb_cuda_training_available(X_train, y_train)

params = {
    "max_depth": 8, "eta": 0.0033968017011338967, "subsample": 0.9731797467870894,
    "colsample_bytree": 0.9095940838020883, "min_child_weight": 1, "gamma": 0.2774238622335642,
    "alpha": 4.2050762518663145, "lambda": 2.5755643313408796,
    "scale_pos_weight": scale_pos_weight, "objective": "binary:logistic", "eval_metric": "logloss",
    **GPU_XGB_PARAMS,
}

dtrain, dval = xgb.DMatrix(X_train, label=y_train), xgb.DMatrix(X_val, label=y_val)
print(f"Training: max_rounds={NUM_BOOST_ROUND}, early_stop={EARLY_STOPPING_ROUNDS}")
booster = xgb.train(params=params, dtrain=dtrain, num_boost_round=NUM_BOOST_ROUND,
                    evals=[(dtrain, "train"), (dval, "val")],
                    early_stopping_rounds=EARLY_STOPPING_ROUNDS,
                    verbose_eval=100)

best_ntree = int(booster.best_iteration + 1) if booster.best_iteration is not None else NUM_BOOST_ROUND
print(f"\nBest iteration: {best_ntree}")

probs_train = predict_probs_booster(booster, X_train, best_ntree)
probs_val   = predict_probs_booster(booster, X_val,   best_ntree)
probs_test  = predict_probs_booster(booster, X_test,  best_ntree)
tr = buy_metrics(y_train, probs_train, BUY_THRESHOLD)
vl = buy_metrics(y_val,   probs_val,   BUY_THRESHOLD)
te = buy_metrics(y_test,  probs_test,  BUY_THRESHOLD)

print(f"Train  logloss={log_loss(y_train,probs_train):.6f}  acc={tr['acc']:.2f}%  P(success|BUY)={tr['buy_success']:.2f}%")
print(f"Val    logloss={log_loss(y_val,  probs_val  ):.6f}  acc={vl['acc']:.2f}%  P(success|BUY)={vl['buy_success']:.2f}%")
print(f"Test   logloss={log_loss(y_test, probs_test ):.6f}  acc={te['acc']:.2f}%  P(success|BUY)={te['buy_success']:.2f}%")

MODEL_PATH = RUN_OUTPUT_DIR / "best_model_xgb.pkl"
joblib.dump({"booster": booster, "best_ntree": best_ntree}, MODEL_PATH)
print(f"Saved -> {MODEL_PATH}")

# Optuna Hyperparameter Search (XGBoost)

In [ ]:
import os, numpy as np, torch, optuna, wandb

optuna.logging.set_verbosity(optuna.logging.WARNING)
os.environ.update({"WANDB_SILENT": "true", "WANDB_CONSOLE": "off"})
os.makedirs(RUN_OUTPUT_DIR / "wandb", exist_ok=True)
wandb_settings = wandb.Settings(silent=True, quiet=True, console="off", root_dir=str(RUN_OUTPUT_DIR / "wandb"))
wandb.login(key="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")

# -- Config --
OPTUNA_N_TRIALS   = 10
OPTUNA_EARLY_STOP = max(10, PATIENCE * 5)
OPTUNA_MAX_ROUNDS = 3000
WANDB_PROJECT, WANDB_GROUP = "NN-Trading-Bot", f"optuna_xgb_{SEED}"

dtrain_opt = xgb.DMatrix(X_train, label=y_train)
dval_opt   = xgb.DMatrix(X_val,   label=y_val)

TUNED_KEYS = ("max_depth", "eta", "subsample", "colsample_bytree", "min_child_weight", "gamma", "alpha", "lambda")
_fmt_val   = lambda v: f"{v:.4g}" if isinstance(v, float) else str(v)
_run_name  = lambda d: "____".join(f"{k}_{_fmt_val(d[k])}" for k in TUNED_KEYS)

def objective(trial):
    hp = {
        "max_depth":        trial.suggest_int("max_depth", 2, 8),
        "eta":              trial.suggest_float("eta", 1e-3, 0.3, log=True),
        "subsample":        trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.4, 1.0),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 20),
        "gamma":            trial.suggest_float("gamma", 0.0, 5.0),
        "alpha":            trial.suggest_float("alpha", 0.0, 10.0),
        "lambda":           trial.suggest_float("lambda", 0.5, 10.0),
    }
    params = {"objective": "binary:logistic", "eval_metric": "logloss", "seed": SEED, "scale_pos_weight": scale_pos_weight, **GPU_XGB_PARAMS, **hp}

    try:
        run = wandb.init(project=WANDB_PROJECT, group=WANDB_GROUP, name=f"trial_{trial.number}__{_run_name(hp)}", config=hp, reinit=True, settings=wandb_settings)
    except Exception:
        run = None

    evals_res = {}
    bst = xgb.train(params=params, dtrain=dtrain_opt, num_boost_round=OPTUNA_MAX_ROUNDS,
                    evals=[(dtrain_opt, "train"), (dval_opt, "eval")],
                    early_stopping_rounds=OPTUNA_EARLY_STOP, evals_result=evals_res, verbose_eval=False)

    best_iter     = int(bst.best_iteration + 1) if bst.best_iteration is not None else OPTUNA_MAX_ROUNDS
    val_logloss   = evals_res["eval"]["logloss"][best_iter - 1]
    train_logloss = evals_res["train"]["logloss"][best_iter - 1]
    trial.set_user_attr("best_ntree", best_iter)
    trial.set_user_attr("val_logloss", val_logloss)

    if run is not None:
        for r, (tr, ev) in enumerate(zip(evals_res["train"]["logloss"], evals_res["eval"]["logloss"])):
            if (r + 1) % 100 == 0:
                wandb.log({"train/logloss": tr, "eval/logloss": ev, "round": r + 1})
        wandb.summary.update({"eval/best_ntree": best_iter, "train/final_logloss": train_logloss, "eval/final_logloss": val_logloss})
        run.finish()

    best_so_far = min((t.value for t in trial.study.trials if t.value is not None), default=val_logloss)
    marker = " *" if val_logloss <= best_so_far else ""
    print(f"  [{trial.number + 1:3d}/{OPTUNA_N_TRIALS}]  train={train_logloss:.6f}  val={val_logloss:.6f}  rounds={best_iter}{marker}")
    return val_logloss

study = optuna.create_study(direction="minimize", study_name="xgb_hparam_search", sampler=optuna.samplers.TPESampler(seed=SEED))
print(f"Starting Optuna search: {OPTUNA_N_TRIALS} trials …")
print(f"{'':>6}{'trial':>8}  {'train_loss':>12}  {'val_loss':>12}  {'rounds':>8}")
print(f"{'':>6}{'-'*8}  {'-'*12}  {'-'*12}  {'-'*8}")
study.optimize(objective, n_trials=OPTUNA_N_TRIALS, show_progress_bar=False)

best_trial  = study.best_trial
best_params = best_trial.params
print(f"\nBest trial #{best_trial.number}  val_logloss={best_trial.value:.6f}")
print("Best params:", best_params)

# -- Final run: retrain on train+val with best params --
final_run = wandb.init(project=WANDB_PROJECT, group=WANDB_GROUP, name=f"BEST_trial_{best_trial.number}__{_run_name(best_params)}", config=best_params, reinit="finish_previous", settings=wandb_settings)

final_params = {"objective": "binary:logistic", "eval_metric": "logloss", "seed": SEED, "scale_pos_weight": scale_pos_weight, **GPU_XGB_PARAMS, **best_params}
best_ntree   = int(best_trial.user_attrs["best_ntree"])
dtrain_full  = xgb.DMatrix(np.concatenate([X_train, X_val]), label=np.concatenate([y_train, y_val]))

print(f"\nRetraining on train+val for {best_ntree} rounds …")
booster = xgb.train(params=final_params, dtrain=dtrain_full, num_boost_round=best_ntree, verbose_eval=False)

probs_test  = predict_probs_booster(booster, X_test,  best_ntree)
probs_train = predict_probs_booster(booster, X_train, best_ntree)
te_opt = buy_metrics(y_test,  probs_test,  BUY_THRESHOLD)
tr_opt = buy_metrics(y_train, probs_train, BUY_THRESHOLD)

print(f"Train  acc={tr_opt['acc']:.2f}%  P(success|BUY)={tr_opt['buy_success']:.2f}%")
print(f"Test   acc={te_opt['acc']:.2f}%  P(success|BUY)={te_opt['buy_success']:.2f}%  logloss={log_loss(y_test, probs_test):.6f}")
wandb.log({"train/final_logloss": log_loss(y_train, probs_train), "train/accuracy": tr_opt["acc"], "train/buy_success": tr_opt["buy_success"],
           "eval/test_logloss": log_loss(y_test, probs_test), "eval/test_accuracy": te_opt["acc"], "eval/test_buy_success": te_opt["buy_success"]})

MODEL_PATH = RUN_OUTPUT_DIR / "best_model_xgb.pkl"
joblib.dump({"booster": booster, "best_ntree": best_ntree}, MODEL_PATH)
print(f"\nSaved -> {MODEL_PATH}")
print(f"Use BUY_THRESHOLD = {BUY_THRESHOLD}")
final_run.finish()
print("W&B runs finished.")